# 02 - Preprocessing: normalizzazione Min-Max e PCA

Applica la normalizzazione Min-Max e la riduzione dimensionale tramite PCA (sezione "Preprocessing: normalizzazione Min-Max e PCA" del Capitolo 7 e sezione "Riduzione della dimensionalità tramite PCA" del Capitolo 6) ai tre dataset acquisiti dal notebook precedente, utilizzando il numero di componenti principali definito in `config.DATASET_CONFIGS`. Le trasformazioni sono calcolate esclusivamente sull'insieme di addestramento.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from config import DATASET_CONFIGS, RANDOM_STATE, TEST_SIZE
from src.data.acquisition import load_dataset
from src.pipeline import load_and_preprocess_dataset

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path.cwd().parent / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
variance_rows = []

for name, cfg in DATASET_CONFIGS.items():
    X, _y = load_dataset(name)
    X_train, X_test, y_train, y_test, pca = load_and_preprocess_dataset(
        name, cfg["n_components"], test_size=TEST_SIZE, random_state=RANDOM_STATE,
    )

    np.savez(
        PROCESSED_DIR / f"{name}.npz",
        X_train=X_train, X_test=X_test,
        y_train=y_train, y_test=y_test,
    )

    variance_rows.append({
        "dataset": name,
        "n_features_originarie": X.shape[1],
        "componenti_selezionate": cfg["n_components"],
        "varianza_spiegata_cumulata": pca.explained_variance_ratio_.sum(),
    })

variance_table = pd.DataFrame(variance_rows)
variance_table.to_csv(TABLES_DIR / "pca_variance.csv", index=False)
variance_table

La tabella corrisponde alla Tabella `pca_componenti` del Capitolo 7 ed è salvata in `results/tables/pca_variance.csv`, così che possa essere rigenerata in modo incrementale senza dover ripetere l'intera campagna sperimentale. I dati preprocessati (feature ridotte ed etichette, separatamente per train e test) sono salvati in `data/processed/<dataset>.npz` e vengono ricaricati dai notebook successivi tramite `numpy.load`.